# Multi-Robot for Airport Luggage Transport

## Portfolio Version

This notebook contains the final implementation of a multi-robot airport luggage transport simulation using Q-learning, PSO tuning, and GWO-based swarm optimization.


### Cell 1 – GUI + Environment (3 Robots, All Modes)

- Builds the **airport grid** (walls, sorting, gates, baggage claim, coffee shop, waiting area).
- Sets up the **GUI**, sidebar, and start menu to choose **Baseline / Optimized / Swarm**.
- Spawns the 3 robots, dynamic obstacles, gate sequence, and runs the **main simulation loop**.
- Handles **bag pipeline logic** (R1: Check-in→Sorting, R2: Sorting→Gate, R3: Gate→Claim) and escape-mode for loops.

In [ ]:
# ===================== CELL 1 – GUI + ENVIRONMENT (3 Robots, Q-Learning) =====================
import sys
import pygame
import numpy as np
import time
import os
import random
import math
import pickle

# --- GLOBAL RANDOM SEED (gates, etc.) ---
GLOBAL_SEED = 42  
RUN_GUI = True   # <-- set to True when running only

# ===================== MODES & Q-LEARNING GLOBALS =====================
MODE_BASELINE  = "BASELINE"
MODE_OPTIMIZED = "OPTIMIZED"
MODE_SWARM     = "SWARM"

# Active mode for the current run (set from the menu)
ACTIVE_MODE = MODE_BASELINE

# Where the trained brains are saved (from Cells 2,5,8)
BASELINE_Q_FILE  = "airport_q_baseline.pkl"
OPTIMIZED_Q_FILE = "airport_q_tables.pkl"
SWARM_Q_FILE     = "airport_swarm_q_tables.pkl"  

q_tables_baseline  = {}
q_tables_optimized = {}
q_tables_swarm     = {}

# ===================== GRID =====================
airport_grid_v11 = np.array([
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], # 0
    [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 1
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 2
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 3
    [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 0, 0, 1], # 4
    [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 3, 3, 3, 3, 4, 4, 4, 1], # 5
    [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 3, 3, 3, 3, 3, 3, 4, 4, 1], # 6
    [1, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 3, 3, 3, 3, 4, 4, 4, 1], # 7
    [1, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 0, 0, 1], # 8
    [1, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 9
    [1, 5, 5, 5, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 10
    [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 11
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 12
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 13
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 14
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 15
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 16
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 17
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 18
    
    [1, 6, 6, 6, 6, 6, 1, 0, 0, 0, 0, 0, 0, 1, 6, 6, 6, 6, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 6, 6, 6, 6, 6, 6, 1], # 19
    [1, 6, 6, 6, 6, 6, 1, 0, 0, 0, 0, 0, 0, 1, 6, 6, 6, 6, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 6, 6, 6, 6, 6, 6, 1], # 20
    [1, 6, 6, 6, 6, 6, 1, 0, 0, 0, 0, 0, 0, 1, 6, 6, 6, 6, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 6, 6, 6, 6, 6, 6, 1], # 21
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]  # 22
])

GRID_HEIGHT, GRID_WIDTH = airport_grid_v11.shape

# ===================== COLORS (UI) =====================
COLOR_FLOOR     = (211, 211, 211)
COLOR_WALL      = (0,   0,   0)
COLOR_CONVEYOR  = (255, 255, 255)
COLOR_CLAIM     = (128, 128, 255)
COLOR_SORTING   = (255, 174, 66)
COLOR_GATES     = (128, 255, 128)
COLOR_ROBOT     = (255,   0,   0)
COLOR_TEXT      = (15,  15,  15)
COLOR_CARD      = (235, 235, 235)
COLOR_BG        = (245, 245, 245)
COLOR_PANEL_BG  = (250, 250, 250)
COLOR_PANEL_BORDER = (120,120,120)
COLOR_RADIO_INACTIVE = (180,180,180)
COLOR_RADIO_ACTIVE   = (40, 40, 40)
COLOR_RESTRICTED = (220, 60, 60)

# Start menu theme
MENU_BG          = (235, 235, 238)
MENU_TEXT        = (10,  30,  80)
MENU_CARD        = (255, 255, 255)
MENU_ACCENT      = (18,  56, 140)
MENU_ACCENT_DARK = (12,  42, 108)

# ===================== DESTINATIONS =====================
START_CHECKIN = (2, 2)     # Check-in
GOAL_SORTING  = (8, 5)     # Sorting entrance (used by Q-table)
SORTING_WAIT_R2 = (8, 4)   # Robot 2 waits INSIDE sorting 
GOAL_GATE_A  = (19, 3)
GOAL_GATE_B  = (20, 16)
GOAL_GATE_C  = (20, 31)
GOAL_CLAIM   = (8, 32)

DEST_BY_NAME = {
    "CHECKIN": START_CHECKIN,
    "SORTING": GOAL_SORTING,
    "GATE_A":  GOAL_GATE_A,
    "GATE_B":  GOAL_GATE_B,
    "GATE_C":  GOAL_GATE_C,
    "CLAIM":   GOAL_CLAIM,
}

GATE_NAMES = ["GATE_A", "GATE_B", "GATE_C"]

# ===================== LOADING Q-TABLES =====================
def load_q_tables(kind):
    """
    kind: 'BASELINE', 'OPTIMIZED', or 'SWARM'
    """
    global q_tables_baseline, q_tables_optimized, q_tables_swarm

    if kind == "BASELINE":
        if q_tables_baseline:
            return True
        try:
            with open(BASELINE_Q_FILE, "rb") as f:
                q_tables_baseline = pickle.load(f)
            print("DEBUG: Loaded baseline Q-tables from", BASELINE_Q_FILE)
            return True
        except FileNotFoundError:
            print(f"ERROR: '{BASELINE_Q_FILE}' not found! Run baseline training first.")
            return False

    elif kind == "OPTIMIZED":
        if q_tables_optimized:
            return True
        try:
            with open(OPTIMIZED_Q_FILE, "rb") as f:
                q_tables_optimized = pickle.load(f)
            print("DEBUG: Loaded optimized Q-tables from", OPTIMIZED_Q_FILE)
            return True
        except FileNotFoundError:
            print(f"ERROR: '{OPTIMIZED_Q_FILE}' not found! Run optimized training first.")
            return False

    elif kind == "SWARM":
        if q_tables_swarm:
            return True
        try:
            with open(SWARM_Q_FILE, "rb") as f:
                q_tables_swarm = pickle.load(f)
            print("DEBUG: Loaded swarm Q-tables from", SWARM_Q_FILE)
            return True
        except FileNotFoundError:
            print(f"ERROR: '{SWARM_Q_FILE}' not found! Run swarm training first.")
            return False

    return False

# ===================== ROBOT / SHARED STATE =====================

# Tiles the ROBOT must not enter (walls + waiting + coffee)
BLOCKED_TILES = {1, 8, 9}

class Robot:
    def __init__(self, name, icon_surface, start_pos):
        self.name = name
        self.icon = icon_surface
        self.pos = list(start_pos)
        self.start_pos = list(start_pos)

        self.state = "IDLE"      # semantic states: see below
        self.goal_name = None    # key in DEST_BY_NAME
        self.carrying = False

        self.cumulative_reward = 0.0
        self.last_step_reward  = 0.0
        self.steps_taken       = 0
        self.legs_completed    = 0

        # loop-escape
        self.pos_history = []
        self.prev_pos = None
        self.stuck_mode_steps_remaining = 0
        self.escape_activations = 0
        self.escape_steps_used  = 0

# Bag stages for communication
BAG_AT_CHECKIN = "AT_CHECKIN"
BAG_WITH_R1    = "WITH_R1"
BAG_AT_SORTING = "AT_SORTING"
BAG_WITH_R2    = "WITH_R2"
BAG_AT_GATE    = "AT_GATE"
BAG_WITH_R3    = "WITH_R3"
BAG_AT_CLAIM   = "AT_CLAIM"

bag_stage = BAG_AT_CHECKIN
current_gate_name = None
cycles_completed = 0

# Global robots (set later after we load GUI & icons)
robot1 = None
robot2 = None
robot3 = None
robots  = []

# Simulation timing / metrics
sim_start_time = None
first_cycle_time = None
total_steps_all = 0

# ========== COMMUNICATION LOG ==========
comm_log = []

def log_comm(msg):
    """Append a short conversational line to the communication log in the sidebar."""
    # 12-hour time with AM/PM (e.g., 3:25 PM)
    ts = time.strftime("%I:%M %p", time.localtime()).lstrip("0")
    comm_log.insert(0, {"time": ts, "msg": msg})
    # Keep it small so sidebar never overflows vertically
    if len(comm_log) > 6:
        comm_log.pop()

# ========== ESCAPE DEBUG LOG (Phase A – point 4) ==========
escape_debug_log = []

def debug_log_escape(rb: Robot, target_pos, q_values):
    """
    Detailed console log whenever escape mode is activated.
    Logs:
      - Mode (Baseline/Optimized/Swarm)
      - Robot name
      - Goal name
      - Position
      - For each action: neighbor tile, blocked?, distance to goal, Q-value
    """
    mode_label = {
        MODE_BASELINE:  "Baseline",
        MODE_OPTIMIZED: "Optimized",
        MODE_SWARM:     "Swarm"
    }.get(ACTIVE_MODE, ACTIVE_MODE)

    r, c = int(rb.pos[0]), int(rb.pos[1])
    entry = {
        "mode": ACTIVE_MODE,
        "robot": rb.name,
        "goal": rb.goal_name,
        "pos": (r, c),
        "target": target_pos,
    }
    escape_debug_log.append(entry)

    print("\n=== ESCAPE DEBUG ===")
    print(f"Mode = {mode_label}")
    print(f"Robot = {rb.name}")
    print(f"Goal  = {rb.goal_name}")
    print(f"Pos   = ({r}, {c}), Target = {target_pos}")

    actions_desc = [("UP", -1, 0), ("DOWN", 1, 0), ("LEFT", 0, -1), ("RIGHT", 0, 1)]
    for idx, (name, dr, dc) in enumerate(actions_desc):
        nr, nc = r + dr, c + dc
        qv = float(q_values[idx])

        if not (0 <= nr < GRID_HEIGHT and 0 <= nc < GRID_WIDTH):
            print(f"  {name}: OUT OF BOUNDS, Q={qv:.3f}")
            continue

        tile = airport_grid_v11[nr, nc]
        dist = abs(nr - target_pos[0]) + abs(nc - target_pos[1])
        blocked = " BLOCKED" if tile in BLOCKED_TILES else ""
        print(f"  {name}: to=({nr},{nc}), tile={tile}{blocked}, dist={dist}, Q={qv:.3f}")

    print("====================\n")

# ===================== DYNAMIC OBSTACLES =====================
USE_DYNAMIC_OBS = True  # press 'D' in RUNNING state to toggle ON/OFF

obstacle1_pos = [12, 10]; obstacle1_dir = 1
obstacle2_pos = [15,  2]; obstacle2_dir = 1
obstacle4_pos = [17, 20]; obstacle4_dir = -1
obstacle6_pos = [ 6, 15]; obstacle6_dir = 1

# High-traffic extras
high_traffic = False
extra4_pos = [ 5, 26]; extra4_dir = 1
extra6_pos = [16, 14]; extra6_dir = 1

def move_dynamic_obstacle(pos, direction, grid, mode='vertical'):
    new_r, new_c = pos[0], pos[1]
    if mode == 'vertical':
        new_r += direction
    elif mode == 'horizontal':
        new_c += direction
    # People can move anywhere except solid walls
    if (0 <= new_r < grid.shape[0] and 0 <= new_c < grid.shape[1]
        and grid[new_r][new_c] != 1):
        return [new_r, new_c], direction
    return pos, -direction

def current_dynamic_set():
    
    if not USE_DYNAMIC_OBS:
        return set()
    s = {
        tuple(obstacle1_pos), tuple(obstacle2_pos),
        tuple(obstacle4_pos), tuple(obstacle6_pos)
    }
    if high_traffic:
        s.update({
            tuple(extra4_pos), tuple(extra6_pos)
        })
    return s

# ===================== LOOP-ESCAPE CONSTANTS =====================
STUCK_HISTORY_LEN      = 10   # look at the last 10 positions
STUCK_OCCURRENCES      = 3    # if current pos appears >= 3 times -> stuck
ESCAPE_MODE_STEPS      = 20   # how many steps we stay in escape mode once triggered

# ===================== SEEDED GATE SEQUENCE =====================
GATE_SEQUENCE_LENGTH = 1000
gate_sequence = []
gate_idx = 0

def reset_gate_sequence():
    """Create a deterministic gate sequence based on the global seed."""
    global gate_sequence, gate_idx
    gate_sequence = [random.choice(GATE_NAMES) for _ in range(GATE_SEQUENCE_LENGTH)]
    gate_idx = 0

def get_next_gate_name():
    """Return the next gate NAME from the pre-generated sequence."""
    global gate_idx
    if not gate_sequence:
        reset_gate_sequence()
    gate = gate_sequence[gate_idx]
    gate_idx = (gate_idx + 1) % GATE_SEQUENCE_LENGTH
    return gate

# ===================== GUI AND DRAWING (ONLY IF RUN_GUI) =====================
if RUN_GUI:
    pygame.init()
    CELL_SIZE   = 30
    GRID_PIX_W  = GRID_WIDTH  * CELL_SIZE
    GRID_PIX_H  = GRID_HEIGHT * CELL_SIZE
    SIDEBAR_W   = 400  # widened sidebar
    SCREEN_WIDTH  = GRID_PIX_W + SIDEBAR_W
    SCREEN_HEIGHT = GRID_PIX_H
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("Airport Luggage Robot Planning - Q-Learning (3 Robots)")
    clock = pygame.time.Clock()

    # Fonts
    label_font         = pygame.font.SysFont(None, 16, bold=True)
    menu_title_font    = pygame.font.SysFont(None, 52, bold=True)
    menu_card_title_font = pygame.font.SysFont(None, 40, bold=True)
    ui_font            = pygame.font.SysFont(None, 20)
    title_font         = pygame.font.SysFont(None, 24, bold=True)
    waiting_font       = pygame.font.SysFont(None, 36, bold=True)
    claim_font         = pygame.font.SysFont(None, 24, bold=True)

    # ===================== ICONS / IMAGES (relative repository paths) =====================
    ASSET_DIR = "assets"
    ROBOT1_ICON_PATH = os.path.join(ASSET_DIR, "robot_1.png")
    ROBOT2_ICON_PATH = os.path.join(ASSET_DIR, "robot_2.png")
    ROBOT3_ICON_PATH = os.path.join(ASSET_DIR, "robot_3.png")

    MAN_ICON_PATH   = os.path.join(ASSET_DIR, "man.png")
    CHAIR1_PATH     = os.path.join(ASSET_DIR, "chairs.png")
    CHAIR2_PATH     = os.path.join(ASSET_DIR, "chairs2.png")
    CHECKIN_PATH    = os.path.join(ASSET_DIR, "checkin.png")
    SORTING_PATH    = os.path.join(ASSET_DIR, "sorting.jpg")
    GATE_PATH       = os.path.join(ASSET_DIR, "gate.png")
    COFFEE_PATH     = os.path.join(ASSET_DIR, "coffee.png")

    def load_image(path, fallback_color=(50,50,50)):
        try:
            if os.path.isfile(path):
                return pygame.image.load(path).convert_alpha()
        except Exception as e:
            print(f"Image load warning for {path}: {e}")
        surf = pygame.Surface((CELL_SIZE, CELL_SIZE), pygame.SRCALPHA)
        pygame.draw.rect(surf, fallback_color, (2,2,CELL_SIZE-4,CELL_SIZE-4))
        return surf

    def load_robot_icon(path):
        img = load_image(path, (255,0,0))
        return pygame.transform.smoothscale(img, (CELL_SIZE, CELL_SIZE))

    # robots icons
    robot1_icon = load_robot_icon(ROBOT1_ICON_PATH)
    robot2_icon = load_robot_icon(ROBOT2_ICON_PATH)
    robot3_icon = load_robot_icon(ROBOT3_ICON_PATH)

    man_icon     = pygame.transform.smoothscale(load_image(MAN_ICON_PATH), (CELL_SIZE, CELL_SIZE))
    chair_icon_1 = load_image(CHAIR1_PATH)
    chair_icon_2 = load_image(CHAIR2_PATH)
    img_checkin  = load_image(CHECKIN_PATH)
    img_sorting  = load_image(SORTING_PATH)
    img_gate     = load_image(GATE_PATH)
    img_coffee   = load_image(COFFEE_PATH)

    def blit_image_cover(dest_surface, image, dest_rect):
        iw, ih = image.get_width(), image.get_height()
        if iw == 0 or ih == 0:
            return
        scale = max(dest_rect.width / iw, dest_rect.height / ih)
        new_w, new_h = int(math.ceil(iw*scale)), int(math.ceil(ih*scale))
        scaled = pygame.transform.smoothscale(image, (new_w, new_h))
        off_x = dest_rect.x + (dest_rect.width  - new_w) // 2
        off_y = dest_rect.y + (dest_rect.height - new_h) // 2
        crop = pygame.Surface((dest_rect.width, dest_rect.height), pygame.SRCALPHA)
        crop.blit(scaled, (off_x - dest_rect.x, off_y - dest_rect.y))
        dest_surface.blit(crop, dest_rect)

    # ===================== MAP DECORATIONS =====================
    SEATING_SIZE = 3
    cr, cc = GRID_HEIGHT // 2, GRID_WIDTH // 2
    for dr in range(-(SEATING_SIZE//2), SEATING_SIZE//2 + 1):
        for dc in range(-(SEATING_SIZE//2), SEATING_SIZE//2 + 1):
            rr = cr + dr
            cc2 = cc + dc
            if 0 <= rr < GRID_HEIGHT and 0 <= cc2 < GRID_WIDTH:
                airport_grid_v11[rr, cc2] = 7

    for r in (5, 6, 7):
        airport_grid_v11[r, 17] = 7
        airport_grid_v11[r, 18] = 7

    for r in (5, 6, 7):
        for c in (13, 14):
            airport_grid_v11[r, c] = 7
    for r in (10, 11, 12):
        for c in (13, 14):
            airport_grid_v11[r, c] = 7

    # Restricted waiting area
    for rr in range(5, 13):
        for cc2 in range(13, 19):
            airport_grid_v11[rr, cc2] = 8

    # Coffee shop obstacle
    for rr in range(11, 16):
        for cc2 in range(29, 34):
            airport_grid_v11[rr, cc2] = 9
    airport_grid_v11[12, 30] = 0
    airport_grid_v11[8, 26] = 4  # baggage claim marking

    # Room / gate rects (Gate C image aligned with new gate area)
    CHECKIN_RECT = pygame.Rect(1 * CELL_SIZE, 1 * CELL_SIZE, 4 * CELL_SIZE, 3 * CELL_SIZE)
    SORTING_RECT = pygame.Rect(1 * CELL_SIZE, 7 * CELL_SIZE, 4 * CELL_SIZE, 4 * CELL_SIZE)
    GATE_A_RECT  = pygame.Rect(2 * CELL_SIZE, 19 * CELL_SIZE, 4 * CELL_SIZE, 3 * CELL_SIZE)
    GATE_B_RECT  = pygame.Rect(14 * CELL_SIZE, 19 * CELL_SIZE, 4 * CELL_SIZE, 3 * CELL_SIZE)
    GATE_C_RECT  = pygame.Rect(29 * CELL_SIZE, 19 * CELL_SIZE, 5 * CELL_SIZE, 3 * CELL_SIZE)
    COFFEE_RECT  = pygame.Rect(29 * CELL_SIZE, 11 * CELL_SIZE, 5 * CELL_SIZE, 5 * CELL_SIZE)

    RESTRICTED_RECT = pygame.Rect(13 * CELL_SIZE, 5 * CELL_SIZE,
                                  (18 - 13 + 1) * CELL_SIZE, (12 - 5 + 1) * CELL_SIZE)

    # ===================== DRAWING FUNCTIONS =====================
    def draw_map():
        for r in range(GRID_HEIGHT):
            for c in range(GRID_WIDTH):
                rect = (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE)
                t = airport_grid_v11[r, c]
                if t == 1:
                    color = COLOR_WALL
                elif t == 3:
                    color = COLOR_CONVEYOR
                elif t == 4:
                    color = COLOR_CLAIM
                elif t == 5:
                    color = COLOR_SORTING
                elif t == 6:
                    color = COLOR_GATES
                elif t == 8:
                    color = COLOR_RESTRICTED
                elif t == 9:
                    color = (160, 120, 90)
                else:
                    color = COLOR_FLOOR
                pygame.draw.rect(screen, color, rect)

    def draw_labels():
        R = CELL_SIZE
        def txt(s): return label_font.render(s, True, (0,0,0))
        screen.blit(txt('CHECK-IN'), (2*R, 2*R))
        screen.blit(txt('SORTING'),  (2*R, 8*R))
        screen.blit(txt('SERVICE HALLWAY'), (13*R, 13.5*R))
        screen.blit(txt('GATE A'),   (3*R, 20*R))
        screen.blit(txt('GATE B'),   (15.5*R, 20*R))
        screen.blit(txt('GATE C'),   (29*R, 20*R))

    def draw_robot_at(pos, icon_surface):
        r, c = pos
        screen.blit(icon_surface, (c * CELL_SIZE, r * CELL_SIZE))

    def draw_dynamic_obstacle(pos):
        r, c = pos
        screen.blit(man_icon, (c * CELL_SIZE, r * CELL_SIZE))

    def draw_room_and_gate_images():
        blit_image_cover(screen, img_checkin, CHECKIN_RECT)
        blit_image_cover(screen, img_sorting, SORTING_RECT)
        blit_image_cover(screen, img_gate,    GATE_A_RECT)
        blit_image_cover(screen, img_gate,    GATE_B_RECT)
        blit_image_cover(screen, img_gate,    GATE_C_RECT)

    def draw_coffee_shop():
        blit_image_cover(screen, img_coffee, COFFEE_RECT)

    def draw_seating_area_center():
        dest_rect = pygame.Rect((cc - SEATING_SIZE // 2) * CELL_SIZE,
                                (cr - SEATING_SIZE // 2) * CELL_SIZE,
                                SEATING_SIZE * CELL_SIZE,
                                SEATING_SIZE * CELL_SIZE)
        blit_image_cover(screen, chair_icon_1, dest_rect)

    def draw_seating_strip_r5to7_c17():
        dest_rect = pygame.Rect(17 * CELL_SIZE, 5 * CELL_SIZE, 2 * CELL_SIZE, 3 * CELL_SIZE)
        blit_image_cover(screen, chair_icon_1, dest_rect)

    def draw_chair_block_r5to7_c13to14():
        dest_rect = pygame.Rect(13 * CELL_SIZE, 5 * CELL_SIZE, 2 * CELL_SIZE, 3 * CELL_SIZE)
        blit_image_cover(screen, chair_icon_2, dest_rect)

    def draw_chair_block_r10to12_c13to14():
        dest_rect = pygame.Rect(13 * CELL_SIZE, 10 * CELL_SIZE, 2 * CELL_SIZE, 3 * CELL_SIZE)
        blit_image_cover(screen, chair_icon_2, dest_rect)

    def draw_waiting_area_label():
        text = "Waiting Area"
        shadow = waiting_font.render(text, True, (30, 30, 30))
        surf   = waiting_font.render(text, True, (245, 245, 245))
        srect  = shadow.get_rect(center=RESTRICTED_RECT.center)
        rect   = surf.get_rect(center=(RESTRICTED_RECT.centerx, RESTRICTED_RECT.centery - 2))
        screen.blit(shadow, srect.move(2, 2))
        screen.blit(surf, rect)

    def draw_baggage_claim_label_centered_span():
        row = 6
        col_start, col_end = 27, 30
        rect = pygame.Rect(col_start * CELL_SIZE, row * CELL_SIZE,
                           (col_end - col_start + 1) * CELL_SIZE, CELL_SIZE)
        text = "Baggage Claim"
        shadow = claim_font.render(text, True, (245, 245, 245))
        surf   = claim_font.render(text, True, (20, 20, 20))
        screen.blit(shadow, shadow.get_rect(center=(rect.centerx+1, rect.centery+1)))
        screen.blit(surf,   surf.get_rect(center=rect.center))

    # Sidebar & stats
    SIDEBAR_RECT = pygame.Rect(GRID_PIX_W, 0, SIDEBAR_W, SCREEN_HEIGHT)

    def draw_sidebar_bg():
        pygame.draw.rect(screen, (248, 248, 252), SIDEBAR_RECT)
        pygame.draw.line(screen, (200, 200, 200), (GRID_PIX_W, 0), (GRID_PIX_W, SCREEN_HEIGHT), 1)

    def manhattan(p1, p2):
        return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

    def get_robot_state_summary(rb: Robot):
        if rb is None:
            return ["Robot: None"]
        carr = "Yes" if rb.carrying else "No"
        return [
            f"{rb.name}: {rb.state}",
            f"  Carrying: {carr}",
            f"  Legs: {rb.legs_completed}, Steps: {rb.steps_taken}",
            f"  Last R: {rb.last_step_reward:+5.1f}",
            f"  Total R: {rb.cumulative_reward:+7.1f}",
        ]

    def get_system_stats_lines():
        elapsed = (time.time() - sim_start_time) if sim_start_time is not None else 0.0
        mode_label = {
            MODE_BASELINE:  "Baseline",
            MODE_OPTIMIZED: "Optimized",
            MODE_SWARM:     "Swarm"
        }.get(ACTIVE_MODE, ACTIVE_MODE)
        return [
            f"Mode: {mode_label}",
            f"Elapsed: {elapsed:5.1f}s",
            f"Cycles completed: {cycles_completed}",
            f"Total steps (all robots): {total_steps_all}",
            f"Current gate: {current_gate_name or '--'}",
            f"Bag stage: {bag_stage}",
            f"Dynamic Obstacles: {'ON' if USE_DYNAMIC_OBS else 'OFF'}",
        ]

    def draw_control_panel():
        # Outer card
        margin = 12
        panel = pygame.Rect(
            GRID_PIX_W + margin,
            margin,
            SIDEBAR_W - 2 * margin,
            SCREEN_HEIGHT - 2 * margin,
        )
        pygame.draw.rect(screen, COLOR_PANEL_BG, panel, border_radius=10)
        pygame.draw.rect(screen, COLOR_PANEL_BORDER, panel, width=1, border_radius=10)

        # Inner content area
        inner_x = panel.x + 16
        inner_w = panel.width - 32
        y = panel.y + 10

        # ---- TITLE ----
        title = title_font.render("Airport Luggage Transportation", True, COLOR_TEXT)
        screen.blit(title, (inner_x, y))
        y += title.get_height() + 8

        # Small subtle separator
        pygame.draw.line(
            screen, (220, 220, 220),
            (inner_x, y), (inner_x + inner_w, y), 1
        )
        y += 10

        # ==== HIGH TRAFFIC TOGGLE ====
        high_rect = pygame.Rect(inner_x, y, 18, 18)
        pygame.draw.rect(
            screen,
            COLOR_RADIO_ACTIVE if high_traffic else COLOR_RADIO_INACTIVE,
            high_rect,
            width=2,
            border_radius=4,
        )
        if high_traffic:
            inner = high_rect.inflate(-6, -6)
            pygame.draw.rect(screen, COLOR_RADIO_ACTIVE, inner, border_radius=3)

        txt = ui_font.render("High Traffic (extra people)", True, COLOR_TEXT)
        screen.blit(txt, (inner_x + 26, y - 2))
        y += max(24, high_rect.height + 10)

        # ==== SYSTEM / MENU AREA ====
        header = ui_font.render("System", True, COLOR_TEXT)
        screen.blit(header, (inner_x, y))
        y += header.get_height() + 6

        # Return-to-menu "button"
        menu_center = (inner_x + 10, y + 9)
        pygame.draw.circle(screen, COLOR_RADIO_INACTIVE, menu_center, 9, width=2)
        txt = ui_font.render("Return to Start Menu (Restart)", True, COLOR_TEXT)
        screen.blit(txt, (inner_x + 26, y))
        y += txt.get_height() + 10

        # ==== SYSTEM STATS ====
        header = ui_font.render("System Stats", True, COLOR_TEXT)
        screen.blit(header, (inner_x, y))
        y += header.get_height() + 4

        for line in get_system_stats_lines():
            line_surf = ui_font.render(line, True, (40, 40, 40))
            screen.blit(line_surf, (inner_x, y))
            y += line_surf.get_height() + 2

        # Hint for dynamic obstacles toggle
        hint_surf = ui_font.render("Hint: Press 'D' to toggle obstacles", True, (120, 120, 120))
        screen.blit(hint_surf, (inner_x, y + 2))
        y += hint_surf.get_height() + 10

        # Small separator before robots
        pygame.draw.line(
            screen, (220, 220, 220),
            (inner_x, y), (inner_x + inner_w, y), 1
        )
        y += 8

        # ==== ROBOT STATS ====
        header = ui_font.render("Robots", True, COLOR_TEXT)
        screen.blit(header, (inner_x, y))
        y += header.get_height() + 4

        for rb in (robot1, robot2, robot3):
            if rb is None:
                continue
            for line in get_robot_state_summary(rb):
                line_surf = ui_font.render(line, True, (40, 40, 40))
                screen.blit(line_surf, (inner_x, y))
                y += line_surf.get_height() + 1
            y += 6  # spacing between robots

        # Separator before communication
        pygame.draw.line(
            screen, (220, 220, 220),
            (inner_x, y), (inner_x + inner_w, y), 1
        )
        y += 8

        # ==== COMMUNICATION LOG ====
        header = ui_font.render("Communication", True, COLOR_TEXT)
        screen.blit(header, (inner_x, y))
        y += header.get_height() + 4

        # Each line: "Robot text" left, time right (12h), all clipped to fit panel
        for entry in comm_log:
            if y > panel.bottom - 24:  # avoid drawing out of panel vertically
                break

            msg = entry["msg"]
            ts  = entry["time"]

            ts_surf = ui_font.render(ts, True, (140, 140, 140))
            max_msg_width = inner_w - ts_surf.get_width() - 12  # space for time + padding

            # Clip message text
            msg_text = msg
            while True:
                msg_surf = ui_font.render(msg_text, True, (60, 60, 60))
                if msg_surf.get_width() <= max_msg_width or len(msg_text) <= 3:
                    break
                # progressively trim and add ellipsis
                msg_text = msg_text[:-4] + "..."

            # Left: message, Right: time
            screen.blit(msg_surf, (inner_x, y))
            screen.blit(
                ts_surf,
                (inner_x + inner_w - ts_surf.get_width(), y)
            )
            y += msg_surf.get_height() + 2

        return panel, high_rect, menu_center

    def point_in_circle(p, center, radius):
        dx = p[0] - center[0]
        dy = p[1] - center[1]
        return dx*dx + dy*dy <= radius*radius

    # ===================== MENU =====================
    STATE_MENU    = "MENU"
    STATE_RUNNING = "RUNNING"
    app_state = STATE_MENU

    def centered_rect(w, h, cx, cy):
        return pygame.Rect(int(cx - w//2), int(cy - h//2), int(w), int(h))

    CARD_W = min(500, int(SCREEN_WIDTH * 0.55))
    CARD_H = 90
    cards = {
        "BASELINE":  centered_rect(CARD_W, CARD_H, SCREEN_WIDTH * 0.50, SCREEN_HEIGHT * 0.35),
        "OPTIMIZED": centered_rect(CARD_W, CARD_H, SCREEN_WIDTH * 0.50, SCREEN_HEIGHT * 0.55),
        "SWARM":     centered_rect(CARD_W, CARD_H, SCREEN_WIDTH * 0.50, SCREEN_HEIGHT * 0.75),
    }

    menu_init_done      = False
    menu_selected_key   = "BASELINE"
    menu_pressed_card   = None
    menu_press_until    = 0.0
    menu_transition_start = None
    menu_pending_choice = None

    def draw_shadow(surf, rect):
        shadow = pygame.Surface((rect.w+18, rect.h+18), pygame.SRCALPHA)
        pygame.draw.rect(shadow, (0,0,0,46), shadow.get_rect(), border_radius=16)
        surf.blit(shadow, (rect.x-9, rect.y-2))

    def card_visual_scale(key, base_rect, now, hover):
        scale = 1.0
        if hover:
            scale = 1.03
        if key == menu_pressed_card and now < menu_press_until:
            scale = 0.98
        w = int(base_rect.w * scale)
        h = int(base_rect.h * scale)
        x = base_rect.centerx - w//2
        y = base_rect.centery - h//2
        return pygame.Rect(x, y, w, h)

    def draw_start_button(label):
        btn_w, btn_h = 260, 50
        rect = pygame.Rect(SCREEN_WIDTH//2 - btn_w//2, int(SCREEN_HEIGHT*0.90), btn_w, btn_h)
        pygame.draw.rect(screen, MENU_ACCENT, rect, border_radius=12)
        pygame.draw.rect(screen, MENU_ACCENT_DARK, rect, width=2, border_radius=12)
        text = ui_font.render(f"Start ({label})", True, (255,255,255))
        screen.blit(text, text.get_rect(center=rect.center))
        return rect

    def draw_menu():
        screen.fill(MENU_BG)
        now = time.time()

        title_surf = menu_title_font.render("Choose Q-Learning Mode", True, MENU_TEXT)
        screen.blit(title_surf, title_surf.get_rect(center=(SCREEN_WIDTH//2, int(SCREEN_HEIGHT*0.18))))

        mx, my = pygame.mouse.get_pos()
        hover_key = None
        for key in ("BASELINE", "OPTIMIZED", "SWARM"):
            base_rect = cards[key]
            hover = base_rect.collidepoint((mx,my))
            if hover:
                hover_key = key
            rect = card_visual_scale(key, base_rect, now, hover)

            draw_shadow(screen, rect)
            card_surf = pygame.Surface((rect.w, rect.h), pygame.SRCALPHA)
            pygame.draw.rect(card_surf, MENU_CARD, card_surf.get_rect(), border_radius=16)
            pygame.draw.rect(card_surf, (206,214,230), card_surf.get_rect(), width=1, border_radius=16)

            if key == "BASELINE":
                title = "Baseline Q-Learning"
            elif key == "OPTIMIZED":
                title = "Optimized Q-Learning"
            else:
                title = "Swarm Mode Q-Learning"

            t_surf = menu_card_title_font.render(title, True, MENU_TEXT)
            card_surf.blit(t_surf, t_surf.get_rect(center=card_surf.get_rect().center))
            screen.blit(card_surf, rect)

        labels = {
            "BASELINE":  "Baseline",
            "OPTIMIZED": "Optimized",
            "SWARM":     "Swarm"
        }
        active_choice = hover_key or menu_selected_key or "BASELINE"
        current_label = labels.get(active_choice, "Baseline")

        start_rect = draw_start_button(current_label)

        if menu_transition_start is not None:
            t = (now - menu_transition_start) / 0.20
            if t >= 1.0:
                start_sim_from_menu(menu_pending_choice)
            else:
                fade_overlay = pygame.Surface((SCREEN_WIDTH, SCREEN_HEIGHT))
                fade_overlay.set_alpha(int(255 * t))
                fade_overlay.fill(MENU_BG)
                screen.blit(fade_overlay, (0,0))

        pygame.display.flip()
        return start_rect, hover_key

    def start_selection(choice_key):
        global menu_pending_choice, menu_transition_start
        menu_pending_choice = choice_key
        menu_transition_start = time.time()

    def handle_menu_events():
        global menu_pressed_card, menu_press_until, menu_selected_key, menu_init_done
        now = time.time()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                raise SystemExit
            if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                pygame.quit()
                raise SystemExit
            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                start_rect, hover_key = draw_menu()
                mx, my = event.pos
                for key, rect in cards.items():
                    if rect.collidepoint((mx,my)):
                        menu_selected_key = key
                        menu_pressed_card = key
                        menu_press_until = now + 0.12
                        pygame.time.set_timer(pygame.USEREVENT+1, 10, loops=1)
                        return
                if start_rect.collidepoint((mx,my)):
                    choice = hover_key or menu_selected_key or "BASELINE"
                    menu_pressed_card = choice
                    menu_press_until = now + 0.12
                    start_selection(choice)
                    return
            if event.type == pygame.USEREVENT+1:
                pass

    # ===================== INITIALIZE SIM FROM MENU =====================
    def start_sim_from_menu(choice_key):
        global app_state, ACTIVE_MODE, sim_start_time, first_cycle_time
        global robot1, robot2, robot3, robots
        global bag_stage, current_gate_name, cycles_completed, total_steps_all
        global obstacle1_pos, obstacle1_dir, obstacle2_pos, obstacle2_dir
        global obstacle4_pos, obstacle4_dir, obstacle6_pos, obstacle6_dir
        global extra4_pos, extra4_dir, extra6_pos, extra6_dir
        global comm_log, USE_DYNAMIC_OBS

        # seed RNGs for reproducible gate order & behaviour
        random.seed(GLOBAL_SEED)
        np.random.seed(GLOBAL_SEED)
        reset_gate_sequence()

        comm_log = []
        USE_DYNAMIC_OBS = True  # start each run with dynamic obstacles ON

        if choice_key == "OPTIMIZED":
            if not load_q_tables("OPTIMIZED"):
                return
            ACTIVE_MODE = MODE_OPTIMIZED
            planner_name = "Optimized Q-Learning"
        elif choice_key == "SWARM":
            if not load_q_tables("SWARM"):
                return
            ACTIVE_MODE = MODE_SWARM
            planner_name = "Swarm Mode Q-Learning"
        else:
            if not load_q_tables("BASELINE"):
                return
            ACTIVE_MODE = MODE_BASELINE
            planner_name = "Baseline Q-Learning"

        # reset dynamic obstacles to original positions
        obstacle1_pos[:] = [12, 10]; obstacle1_dir = 1
        obstacle2_pos[:] = [15,  2]; obstacle2_dir = 1
        obstacle4_pos[:] = [17, 20]; obstacle4_dir = -1
        obstacle6_pos[:] = [ 6, 15]; obstacle6_dir = 1
        extra4_pos[:] = [5, 26]; extra4_dir = 1
        extra6_pos[:] = [16,14]; extra6_dir = 1

        # Create robots
        robot1 = Robot("Robot 1", robot1_icon, START_CHECKIN)
        robot2 = Robot("Robot 2", robot2_icon, SORTING_WAIT_R2)  # wait INSIDE sorting
        robot3 = Robot("Robot 3", robot3_icon, GOAL_CLAIM)       # wait near claim

        robots = [robot1, robot2, robot3]

        # Initial pipeline: new bag at check-in, R1 takes it to sorting
        bag_stage = BAG_WITH_R1
        current_gate_name = None
        cycles_completed = 0
        total_steps_all = 0
        first_cycle_time = None

        robot1.state = "TO_SORTING"
        robot1.goal_name = "SORTING"
        robot1.carrying = True

        robot2.state = "WAITING_AT_SORTING"
        robot2.goal_name = None
        robot2.carrying = False

        robot3.state = "WAITING_AT_CLAIM"
        robot3.goal_name = None
        robot3.carrying = False

        sim_start_time = time.time()
        app_state = STATE_RUNNING
        log_comm(f"System: {planner_name} on, R1 has first bag.")
        print(f"DEBUG: Starting sim with {planner_name}")

        # end transition
        global menu_transition_start, menu_pending_choice
        menu_transition_start = None
        menu_pending_choice = None

    # ===================== PANEL CLICK HANDLER =====================
    def handle_panel_click(mouse_pos, panel, high_rect, menu_center):
        global high_traffic, app_state, menu_init_done
        mx, my = mouse_pos

        # Toggle high traffic
        if high_rect.collidepoint(mouse_pos) or pygame.Rect(high_rect.x, high_rect.y-6,
                                                            panel.w-20, 28).collidepoint(mouse_pos):
            high_traffic = not high_traffic
            state = "ON" if high_traffic else "OFF"
            log_comm(f"System: High traffic {state}.")
            return

        # Return to menu
        if point_in_circle((mx,my), menu_center, 14) or pygame.Rect(panel.x+8, menu_center[1]-14,
                                                                    panel.w-16, 28).collidepoint(mouse_pos):
            print("DEBUG MENU: Return to menu requested.")
            app_state = STATE_MENU
            menu_init_done = False
            return

    # ===================== ROBOT MOTION WITH Q-LEARNING =====================
    def move_robot_one_step(rb: Robot):
        """
        One Q-learning driven step for a robot, if it currently has a goal_name.
        Includes loop-escape logic and dynamic obstacle + other-robot avoidance.
        """
        global total_steps_all

        if rb.goal_name is None:
            rb.last_step_reward = 0.0
            return

        # choose active Q table based on ACTIVE_MODE
        if ACTIVE_MODE == MODE_BASELINE:
            active_tables = q_tables_baseline
        elif ACTIVE_MODE == MODE_OPTIMIZED:
            active_tables = q_tables_optimized
        else:
            active_tables = q_tables_swarm

        if rb.goal_name not in active_tables:
            rb.last_step_reward = 0.0
            return

        brain = active_tables[rb.goal_name]
        target_pos = DEST_BY_NAME[rb.goal_name]

        r, c = int(rb.pos[0]), int(rb.pos[1])
        rows, cols = airport_grid_v11.shape
        if not (0 <= r < rows and 0 <= c < cols):
            rb.last_step_reward = 0.0
            return

        q_values = brain[r, c]

        # dynamic obstacles + other robots as avoidance
        dynamic_obstacles = current_dynamic_set()
        other_robot_positions = {tuple(r_o.pos) for r_o in robots if r_o is not rb}
        forbidden = dynamic_obstacles.union(other_robot_positions)

        goal_r, goal_c = target_pos
        actions_order = list(np.argsort(q_values)[::-1])

        # --- loop detection per robot ---
        current_pos_tuple = tuple(rb.pos)
        rb.pos_history.append(current_pos_tuple)
        if len(rb.pos_history) > STUCK_HISTORY_LEN:
            rb.pos_history.pop(0)

        if rb.pos_history.count(current_pos_tuple) >= STUCK_OCCURRENCES and rb.stuck_mode_steps_remaining == 0:
            rb.stuck_mode_steps_remaining = ESCAPE_MODE_STEPS
            rb.escape_activations += 1
            log_comm(f"{rb.name}: Escape mode near {rb.goal_name}.")
            print(f"DEBUG: Escape mode activated for {rb.name} at {current_pos_tuple}")
            # Phase A: detailed escape debug log (Q-values, distances, tiles)
            debug_log_escape(rb, target_pos, q_values)

        in_escape_mode = (rb.stuck_mode_steps_remaining > 0)
        if in_escape_mode:
            rb.stuck_mode_steps_remaining -= 1
            rb.escape_steps_used += 1

        valid_neighbors = []
        for action in actions_order:
            dr, dc = [(-1,0), (1,0), (0,-1), (0,1)][action]
            nr, nc = r + dr, c + dc

            if not (0 <= nr < rows and 0 <= nc < cols):
                continue

            tile = airport_grid_v11[nr][nc]
            if tile in BLOCKED_TILES:
                continue
            if (nr, nc) in forbidden:
                continue

            visit_count = rb.pos_history.count((nr, nc))
            dist = abs(nr - goal_r) + abs(nc - goal_c)

            valid_neighbors.append({
                "pos": (nr, nc),
                "action": action,
                "q": float(q_values[action]),
                "dist": dist,
                "visits": visit_count,
            })

        chosen_pos = None

        if valid_neighbors:
            if in_escape_mode:
                candidates = valid_neighbors
                if rb.prev_pos is not None:
                    non_back = [n for n in candidates if n["pos"] != rb.prev_pos]
                    if non_back:
                        candidates = non_back

                min_visits = min(n["visits"] for n in candidates)
                candidates = [n for n in candidates if n["visits"] == min_visits]

                min_dist = min(n["dist"] for n in candidates)
                candidates = [n for n in candidates if n["dist"] == min_dist]

                best = max(candidates, key=lambda n: n["q"])
                chosen_pos = best["pos"]
            else:
                best = max(valid_neighbors, key=lambda n: n["q"])
                chosen_pos = best["pos"]

        if chosen_pos is not None:
            old_dist = abs(current_pos_tuple[0] - goal_r) + abs(current_pos_tuple[1] - goal_c)
            new_dist = abs(chosen_pos[0]      - goal_r) + abs(chosen_pos[1]      - goal_c)
            # shaped step reward for metrics only
            step_reward = -1 + 0.5 * (old_dist - new_dist)
            if chosen_pos == target_pos and rb.carrying:
                step_reward += 100.0

            rb.cumulative_reward += step_reward
            rb.last_step_reward  = step_reward

            rb.prev_pos = current_pos_tuple
            rb.pos[:] = [chosen_pos[0], chosen_pos[1]]
            rb.steps_taken += 1
            total_steps_all += 1
        else:
            rb.last_step_reward = 0.0

    # ===================== BAG FLOW / ROBOT COORDINATION =====================
    def update_bag_and_states():
        """
        Implements the 3-robot pipeline:
        R1: Check-In → Sorting (handover to R2)
        R2: Sorting → Gate (handover to R3)
        R3: Gate → Claim

        FIX: new bag is only released when:
             - previous bag is already at claim (BAG_AT_CLAIM)
             - R1 is at Check-In
             - R2 is waiting at Sorting
        This prevents R1 from leaving Check-In early.
        """
        global bag_stage, current_gate_name, cycles_completed, first_cycle_time

        # --- R1 -> R2 handover at sorting (Robot2 waits INSIDE sorting) ---
        if bag_stage == BAG_WITH_R1 and robot1.carrying:
            if manhattan(robot1.pos, robot2.pos) <= 1:
                # complete handover
                bag_stage = BAG_WITH_R2
                robot1.carrying = False
                robot2.carrying = True
                robot1.legs_completed += 1

                # choose gate for this bag
                current_gate_name = get_next_gate_name()

                # R1 goes back to check-in and waits
                robot1.state = "RETURN_TO_CHECKIN"
                robot1.goal_name = "CHECKIN"

                # R2 leaves sorting to gate
                robot2.state = f"TO_{current_gate_name}"
                robot2.goal_name = current_gate_name

                log_comm(f"R1: Bag at Sorting, R2 to {current_gate_name}.")
                print(f"DEBUG: R1->R2 handover, gate = {current_gate_name}")

        # --- R2 arrives at gate with bag ---
        if bag_stage == BAG_WITH_R2 and robot2.carrying and current_gate_name is not None:
            gate_pos = DEST_BY_NAME[current_gate_name]
            if robot2.pos == list(gate_pos):
                bag_stage = BAG_AT_GATE
                robot2.carrying = False
                robot2.legs_completed += 1
                robot2.state = "WAITING_AT_GATE"
                robot2.goal_name = None
                log_comm(f"R2: At {current_gate_name}, bag ready for R3.")

        # --- R3 starts moving from CLAIM to gate when bag is at gate ---
        if bag_stage == BAG_AT_GATE and not robot3.carrying:
            if robot3.state == "WAITING_AT_CLAIM":
                robot3.state = f"TO_{current_gate_name}"
                robot3.goal_name = current_gate_name
                log_comm(f"R3: Moving to {current_gate_name} for pickup.")

        # --- R3 meets R2 at gate, handover gate -> R3 ---
        if bag_stage == BAG_AT_GATE and current_gate_name is not None:
            if manhattan(robot2.pos, robot3.pos) <= 1:
                bag_stage = BAG_WITH_R3
                robot3.carrying = True
                robot3.legs_completed += 1
                robot3.state = "TO_CLAIM"
                robot3.goal_name = "CLAIM"

                # R2 returns to Sorting waiting position (inside)
                robot2.state = "RETURN_TO_SORTING"
                robot2.goal_name = "SORTING"
                log_comm(f"R3: Got bag at {current_gate_name}, heading to Claim.")
                log_comm("R2: Returning to Sorting.")

        # --- R3 reaches baggage claim with bag: mark cycle completed, but DON'T spawn next bag yet ---
        if bag_stage == BAG_WITH_R3 and robot3.carrying:
            if robot3.pos == list(GOAL_CLAIM):
                bag_stage = BAG_AT_CLAIM
                robot3.carrying = False
                log_comm("R3: Bag delivered at Claim. Cycle done.")
                cycles_completed += 1
                if first_cycle_time is None and sim_start_time is not None:
                    first_cycle_time = time.time() - sim_start_time
                    print(f"DEBUG: First end-to-end cycle completed in {first_cycle_time:.2f}s")

                # R3 stays at claim and waits for next cycle
                robot3.state = "WAITING_AT_CLAIM"
                robot3.goal_name = None

        # --- Parking behaviours / final positions ---

        # If R1 has returned to check-in and not carrying, keep him idle there
        if robot1.state.startswith("RETURN_TO_CHECKIN") and tuple(robot1.pos) == START_CHECKIN:
            if bag_stage != BAG_WITH_R1:
                robot1.state = "IDLE_AT_CHECKIN"
                robot1.goal_name = None

        # If R2 has returned to sorting door, park him as waiting at sorting
        if robot2.state.startswith("RETURN_TO_SORTING") and tuple(robot2.pos) == GOAL_SORTING:
            robot2.state = "WAITING_AT_SORTING"
            robot2.goal_name = None

        # --- START NEXT CYCLE ONLY WHEN R1 & R2 ARE BOTH READY ---
        if bag_stage == BAG_AT_CLAIM:
            if (tuple(robot1.pos) == START_CHECKIN and
                robot2.state == "WAITING_AT_SORTING"):

                bag_stage = BAG_WITH_R1
                robot1.carrying = True
                robot1.state = "TO_SORTING"
                robot1.goal_name = "SORTING"
                log_comm("System: New bag at Check-In, R1 go.")
            else:
                if (not robot1.carrying and
                    robot1.state not in ("RETURN_TO_CHECKIN", "IDLE_AT_CHECKIN") and
                    robot1.goal_name != "CHECKIN"):
                    robot1.state = "RETURN_TO_CHECKIN"
                    robot1.goal_name = "CHECKIN"
                    log_comm("System: Waiting for R1 at Check-In.")

    # ===================== MAIN LOOP =====================
    running = True
    print("Window ready. Choose Q-Learning mode in the menu.")
    
    while running:
        if app_state == STATE_MENU:
            if not menu_init_done:
                menu_pressed_card = None
                menu_press_until = 0.0
                menu_transition_start = None
                menu_pending_choice = None
                menu_selected_key = "BASELINE"
                menu_init_done = True
            handle_menu_events()
            draw_menu()
            clock.tick(60)
            continue

        # --- RUNNING STATE ---
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                elif event.key == pygame.K_d:
                    # Phase A: toggle dynamic obstacles ON/OFF at runtime
                    USE_DYNAMIC_OBS = not USE_DYNAMIC_OBS
                    state_str = "ON" if USE_DYNAMIC_OBS else "OFF"
                    print(f"DEBUG: Dynamic obstacles set to {state_str}")
                    log_comm(f"System: Obstacles {state_str}.")
            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                panel, high_rect, menu_center = draw_control_panel()
                handle_panel_click(event.pos, panel, high_rect, menu_center)

        # Move dynamic obstacles (only if enabled)
        if USE_DYNAMIC_OBS:
            obstacle1_pos[:], obstacle1_dir = move_dynamic_obstacle(obstacle1_pos, obstacle1_dir,
                                                                    airport_grid_v11, mode='vertical')
            obstacle2_pos[:], obstacle2_dir = move_dynamic_obstacle(obstacle2_pos, obstacle2_dir,
                                                                    airport_grid_v11, mode='horizontal')
            obstacle4_pos[:], obstacle4_dir = move_dynamic_obstacle(obstacle4_pos, obstacle4_dir,
                                                                    airport_grid_v11, mode='vertical')
            obstacle6_pos[:], obstacle6_dir = move_dynamic_obstacle(obstacle6_pos, obstacle6_dir,
                                                                    airport_grid_v11, mode='horizontal')

            if high_traffic:
                extra4_pos[:], extra4_dir = move_dynamic_obstacle(extra4_pos, extra4_dir,
                                                                  airport_grid_v11, mode='vertical')
                extra6_pos[:], extra6_dir = move_dynamic_obstacle(extra6_pos, extra6_dir,
                                                                  airport_grid_v11, mode='vertical')

        # --- ROBOTS MOVE (only if they currently have goals) ---
        for rb in robots:
            if rb.goal_name is not None:
                move_robot_one_step(rb)

        # --- BAG PIPELINE LOGIC (handoffs & cycle control) ---
        update_bag_and_states()

        # --- DRAW EVERYTHING ---
        screen.fill(COLOR_FLOOR)
        draw_map()
        draw_labels()
        draw_room_and_gate_images()
        draw_seating_area_center()
        draw_seating_strip_r5to7_c17()
        draw_chair_block_r5to7_c13to14()
        draw_chair_block_r10to12_c13to14()
        draw_coffee_shop()
        draw_waiting_area_label()
        draw_baggage_claim_label_centered_span()
        draw_sidebar_bg()

        # dynamic obstacles
        if USE_DYNAMIC_OBS:
            for pos in (obstacle1_pos, obstacle2_pos, obstacle4_pos, obstacle6_pos):
                draw_dynamic_obstacle(pos)
            if high_traffic:
                for pos in (extra4_pos, extra6_pos):
                    draw_dynamic_obstacle(pos)

        # robots
        if robot1 is not None:
            draw_robot_at(robot1.pos, robot1.icon)
        if robot2 is not None:
            draw_robot_at(robot2.pos, robot2.icon)
        if robot3 is not None:
            draw_robot_at(robot3.pos, robot3.icon)

        panel, high_rect, menu_center = draw_control_panel()

        # planner label
        if ACTIVE_MODE == MODE_BASELINE:
            planner_name = "Baseline Q-Learning"
        elif ACTIVE_MODE == MODE_OPTIMIZED:
            planner_name = "Optimized Q-Learning"
        else:
            planner_name = "Swarm Mode Q-Learning"

        planner_tag = label_font.render(planner_name, True, (0,0,0))
        screen.blit(planner_tag, (8, SCREEN_HEIGHT - 24))

        # Mouse hover tile info
        mx, my = pygame.mouse.get_pos()
        if mx < GRID_PIX_W and my < GRID_PIX_H:
            c = mx // CELL_SIZE
            r = my // CELL_SIZE
            hover_text = f"Tile: ({r} x {c})"
        else:
            hover_text = "Tile: (-- x --)"

        hover_surf = ui_font.render(hover_text, True, (40, 40, 40))
        screen.blit(hover_surf, (GRID_PIX_W + 16, SCREEN_HEIGHT - 24))

        pygame.display.flip()
        clock.tick(10)  # slower runtime

    pygame.quit()


### Cell 2 – Baseline Q-Learning Training

- Trains a **simple baseline Queue-learning** policy for each destination (Sorting, all Gates, Claim, Check-in).
- Uses plain rewards: **+100 on goal, −1 per step, −10 on invalid moves**; fewer episodes and simpler exploration.
- Saves all baseline Q-tables to **`airport_q_baseline.pkl`** for later testing.

In [ ]:
# =========================
# CELL 2 – BASELINE Q-LEARNING TRAINING
# =========================

import numpy as np
import random
import pickle

# Destinations used in training

DESTINATIONS = {
    "SORTING":  (8, 5),
    "GATE_A":   (19, 3),
    "GATE_B":   (20, 16),
    "GATE_C":   (20, 31),   
    "CLAIM":    (8, 32),
    "CHECKIN":  (2, 2),
}

def train_baseline(target_pos, name, alpha=0.5, gamma=0.89, episodes=2100):
    """
    Baseline Q-learning:
      - Simple reward: +100 on goal, -1 per step, -10 on invalid / blocked.
      - Fewer episodes than optimized.
      - Lower gamma so it is less farsighted.
      - Higher min_epsilon so it keeps more randomness.
    This intentionally makes the baseline weaker but still functional.
    """
    rows, cols = 23, 35  
    q_table = np.zeros((rows, cols, 4))

    # Epsilon-greedy parameters 
    epsilon = 1.0
    min_epsilon = 0.08
    decay = 0.995

    print(f"Training baseline brain for {name} at {target_pos} ...")

    for ep in range(episodes):
        
        start_r = ep % rows
        start_c = (ep // rows) % cols
        r, c = start_r, start_c

        done = False
        steps = 0
        max_steps = 500  # safety cap

        while not done and steps < max_steps:
            steps += 1

            # Epsilon-greedy action selection
            if random.random() < epsilon:
                action = random.randint(0, 3)
            else:
                action = int(np.argmax(q_table[r, c]))

            # Map action to movement
            if action == 0:   # up
                nr, nc = r - 1, c
            elif action == 1: # down
                nr, nc = r + 1, c
            elif action == 2: # left
                nr, nc = r, c - 1
            else:             # right
                nr, nc = r, c + 1

            # Check bounds
            if nr < 0 or nr >= rows or nc < 0 or nc >= cols:
                # Invalid move (outside grid)
                reward = -10
                nr, nc = r, c  # stay in place
            else:
                tile = airport_grid_v11[nr][nc]

                # Walls / blocked tiles
                if tile in BLOCKED_TILES:
                    reward = -10
                    nr, nc = r, c  # stay in place
                else:
                    # Goal check
                    if (nr, nc) == target_pos:
                        reward = +100
                        done = True
                    else:
                        # Small step penalty
                        reward = -1

            # Q-learning update
            old_value = q_table[r, c, action]
            next_max = np.max(q_table[nr, nc])

            new_value = old_value + alpha * (reward + gamma * next_max - old_value)
            q_table[r, c, action] = new_value

            # Move to next state
            r, c = nr, nc

        # Decay epsilon each episode
        epsilon = max(min_epsilon, epsilon * decay)

        if (ep + 1) % 100 == 0:
            print(f"  Episode {ep+1}/{episodes} for {name} - epsilon={epsilon:.3f}")

    print(f"Training {name} completed.\n")
    return q_table

baseline_q_tables = {}
print("=== Baseline training started ===\n")
for name, pos in DESTINATIONS.items():
    baseline_q_tables[name] = train_baseline(pos, name)

with open("airport_q_baseline.pkl", "wb") as f:
    pickle.dump(baseline_q_tables, f)

print("=== All baseline trainings completed ===")
print("Baseline Q-tables saved to 'airport_q_baseline.pkl'")


### Cell 3 – Baseline Policy Testing (Offline)

- Loads **baseline Q-tables** from `airport_q_baseline.pkl`.
- Greedily tests trips like **Check-in → Gate → Check-in** and **Gate → Claim** without running the GUI.
- Prints whether each path **succeeds** and how many **steps** it takes.

In [ ]:
# =========================
# CELL 3 – BASELINE TESTING (offline diagnostic)
# =========================

import numpy as np
import pickle

# Destinations 
DESTINATIONS = {
    "SORTING":  (8, 5),
    "GATE_A":   (19, 3),
    "GATE_B":   (20, 16),
    "GATE_C":   (20, 31),   
    "CLAIM":    (8, 32),
    "CHECKIN":  (2, 2),
}

# Load baseline tables
with open("airport_q_baseline.pkl", "rb") as f:
    baseline_q_tables = pickle.load(f)

def test_trip(start_pos, name, max_steps=200):
    brain = baseline_q_tables[name]
    target_pos = DESTINATIONS[name]
    r, c = start_pos
    steps = 0

    while steps < max_steps:
        action = int(np.argmax(brain[r, c]))
        dr, dc = [(-1,0), (1,0), (0,-1), (0,1)][action]
        nr, nc = r + dr, c + dc

        if (nr < 0 or nr >= 23 or
            nc < 0 or nc >= 35 or
            airport_grid_v11[nr, nc] in BLOCKED_TILES):
            return -1  # invalid move

        r, c = nr, nc
        steps += 1

        if (r, c) == target_pos:
            return steps

    return -1  # timeout

# === SCENARIO 1: Departure (Check-In -> Any Gate -> Check-In) ===
print("=== BASELINE TEST: SCENARIO 1 (Departure: Check-In → Gate → Check-In) ===")
gates = ["GATE_A", "GATE_B", "GATE_C"]

for i, gate_name in enumerate(gates):
    steps = test_trip(DESTINATIONS["CHECKIN"], gate_name)
    if steps != -1:
        print(f"Test {i+1}: Check-In -> {gate_name} | Steps = {steps} (Success)")
        back_steps = test_trip(DESTINATIONS[gate_name], "CHECKIN")
        if back_steps != -1:
            print(f"        {gate_name} -> Check-In | Steps = {back_steps} (Success)")
        else:
            print(f"        {gate_name} -> Check-In | FAILED")
    else:
        print(f"Test {i+1}: Check-In -> {gate_name} | FAILED")

# === SCENARIO 2: Arrival (Gate -> Claim) ===
print("\n=== BASELINE TEST: SCENARIO 2 (Arrival: Gate → Claim) ===")
for i, gate_name in enumerate(gates):
    steps = test_trip(DESTINATIONS[gate_name], "CLAIM")
    if steps != -1:
        print(f"Test {i+1}: {gate_name} -> Baggage Claim | Steps = {steps} (Success)")
    else:
        print(f"Test {i+1}: {gate_name} -> Baggage Claim | FAILED")


### Cell 4 – PSO Hyperparameter Optimization (Optimized Mode)

- Uses **Particle Swarm Optimization (PSO)** to search for good Q-learning hyperparameters  
  (α, γ, ε-decay, min-ε) for gate navigation.
- Evaluates each particle using distance-shaped rewards and **average return** over the last episodes.
- Saves the best hyperparameters into **`best_pso_params.pkl`**.

In [ ]:
# =========================
# CELL 4 – PSO OPTIMIZATION (Particle Swarm Optimizer for hyperparameters)
# =========================

import numpy as np
import random
import pickle


HYPERPARAM_SPACE = {
    "alpha":         (0.1, 1.0),
    "gamma":         (0.80, 0.999),
    "epsilon_decay": (0.995, 0.9999),
    "min_epsilon":   (0.01, 0.2)
}


DESTINATIONS = {
    "SORTING":  (8, 5),
    "GATE_A":   (19, 3),
    "GATE_B":   (20, 16),
    "GATE_C":   (20, 31),
    "CLAIM":    (8, 32),
    "CHECKIN":  (2, 2),
}

def evaluate_hyperparameters(params):
    """
    Return average reward across 3 gates using given hyperparameters.
    Uses the same distance-shaped reward scheme as the optimized training.
    """
    alpha, gamma, eps_decay, min_eps = params

    test_targets = ["GATE_A", "GATE_B", "GATE_C"]
    episodes = 800
    max_steps = 200

    rows, cols = airport_grid_v11.shape
    accumulated_score = 0.0

    for target_name in test_targets:
        target_pos = DESTINATIONS[target_name]
        q_table = np.zeros((rows, cols, 4))
        epsilon = 1.0
        total_rewards = []

        for ep in range(episodes):
            # random valid start
            while True:
                r, c = random.randint(0, rows - 1), random.randint(0, cols - 1)
                if airport_grid_v11[r, c] not in BLOCKED_TILES:
                    break

            done = False
            ep_reward = 0

            for step in range(max_steps):
                if random.random() < epsilon:
                    action = random.randint(0, 3)
                else:
                    action = int(np.argmax(q_table[r, c]))

                dr, dc = [(-1, 0), (1, 0), (0, -1), (0, 1)][action]
                nr, nc = r + dr, c + dc

                old_dist = abs(r - target_pos[0]) + abs(c - target_pos[1])

                if (nr < 0 or nr >= rows or
                    nc < 0 or nc >= cols or
                    airport_grid_v11[nr, nc] in BLOCKED_TILES):
                    reward = -10
                    nr, nc = r, c
                    new_dist = old_dist
                elif (nr, nc) == target_pos:
                    reward = 100
                    done = True
                    new_dist = 0
                else:
                    new_dist = abs(nr - target_pos[0]) + abs(nc - target_pos[1])
                    reward = -1
                    delta_d = old_dist - new_dist
                    reward += 0.5 * delta_d

                old_val = q_table[r, c, action]
                next_max = np.max(q_table[nr, nc])
                q_table[r, c, action] = (1 - alpha) * old_val + alpha * (reward + gamma * next_max)

                r, c = nr, nc
                ep_reward += reward

                if done:
                    break

            epsilon = max(min_eps, epsilon * eps_decay)
            total_rewards.append(ep_reward)

        accumulated_score += np.mean(total_rewards[-50:])

    return accumulated_score / len(test_targets)


def sample_random_params():
    """Sample one random point from the hyperparameter space."""
    a = np.random.uniform(*HYPERPARAM_SPACE["alpha"])
    g = np.random.uniform(*HYPERPARAM_SPACE["gamma"])
    d = np.random.uniform(*HYPERPARAM_SPACE["epsilon_decay"])
    m = np.random.uniform(*HYPERPARAM_SPACE["min_epsilon"])
    return np.array([a, g, d, m], dtype=float)


def clamp_params(vec):
    """Clamp a position vector to the valid hyperparameter bounds."""
    vec[0] = np.clip(vec[0], *HYPERPARAM_SPACE["alpha"])
    vec[1] = np.clip(vec[1], *HYPERPARAM_SPACE["gamma"])
    vec[2] = np.clip(vec[2], *HYPERPARAM_SPACE["epsilon_decay"])
    vec[3] = np.clip(vec[3], *HYPERPARAM_SPACE["min_epsilon"])
    return vec


def PSO_optimize(num_particles=10, iterations=10):
    """
    Particle Swarm Optimization for Q-learning hyperparameters.
    Tries to maximize the fitness returned by evaluate_hyperparameters().
    Pattern matches the Lab approach.
    """
    print("=== PSO Optimization for Q-learning Hyperparameters ===\n")

    # PSO coefficients
    w  = 0.5
    c1 = 1.5
    c2 = 1.5

    dim = 4

    # Initialize swarm and velocity
    swarm = np.zeros((num_particles, dim), dtype=float)
    velocity = np.zeros((num_particles, dim), dtype=float)

    for i in range(num_particles):
        swarm[i] = sample_random_params()

    # Personal bests
    personal_best = swarm.copy()
    personal_best_fitness = np.array([evaluate_hyperparameters(p) for p in swarm], dtype=float)

    # Global best
    best_idx = int(np.argmax(personal_best_fitness))
    global_best = personal_best[best_idx].copy()
    global_best_fitness = float(personal_best_fitness[best_idx])

    # Main PSO loop
    for it in range(iterations):
        print(f"Iteration {it + 1}/{iterations}")

        for i in range(num_particles):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            velocity[i] = (
                w * velocity[i]
                + c1 * r1 * (personal_best[i] - swarm[i])
                + c2 * r2 * (global_best - swarm[i])
            )

            swarm[i] = swarm[i] + velocity[i]
            swarm[i] = clamp_params(swarm[i])

            fitness = evaluate_hyperparameters(swarm[i])

            # Update personal best
            if fitness > personal_best_fitness[i]:
                personal_best[i] = swarm[i].copy()
                personal_best_fitness[i] = fitness

        # Update global best
        best_idx = int(np.argmax(personal_best_fitness))
        if personal_best_fitness[best_idx] > global_best_fitness:
            global_best = personal_best[best_idx].copy()
            global_best_fitness = float(personal_best_fitness[best_idx])

        print(f" Best avg reward so far: {global_best_fitness:.2f}")
        print(f" Best params: {global_best}\n")

    print("=== PSO Optimization Finished ===")
    print("Best Hyperparameters Found (global best):")
    print(f"alpha:         {global_best[0]:.4f}")
    print(f"gamma:         {global_best[1]:.4f}")
    print(f"epsilon_decay: {global_best[2]:.5f}")
    print(f"min_epsilon:   {global_best[3]:.3f}")
    print(f"Avg Reward:    {global_best_fitness:.2f}")

    return global_best, global_best_fitness


best_params, best_reward = PSO_optimize()

# === SAVE BEST PARAMS SO OTHER CELLS CAN USE THEM AUTOMATICALLY ===
with open("best_pso_params.pkl", "wb") as f:
    pickle.dump(best_params, f)

print("\nSaved best PSO parameters to 'best_pso_params.pkl':")
print(" alpha        =", best_params[0])
print(" gamma        =", best_params[1])
print(" epsilon_decay=", best_params[2])
print(" min_epsilon  =", best_params[3])


### Cell 5 – Optimized Q-Learning Training (Using PSO Params)

- Loads the best PSO hyperparameters from **`best_pso_params.pkl`**
- Trains **optimized Q-tables** for all destinations with:
  - same base rewards as baseline, plus **distance-based shaping** to encourage progress.
- Saves the resulting optimized brains to **`airport_q_tables.pkl`**.

In [ ]:
# =========================
# CELL 5 – TRAIN OPTIMIZED BRAINS (USING BEST PARAMS)
# =========================

import numpy as np
import random
import pickle
import os

# ===================== LOAD OPTIMIZED HYPERPARAMS FROM PSO =====================

if os.path.exists("best_pso_params.pkl"):
    with open("best_pso_params.pkl", "rb") as f:
        best_params = pickle.load(f)

    # Safety: ensure correct shape
    if len(best_params) != 4:
        raise ValueError("best_pso_params.pkl must contain exactly 4 values: alpha, gamma, epsilon_decay, min_epsilon")

    OPT_ALPHA, OPT_GAMMA, OPT_DECAY, OPT_MIN_EPS = best_params

    print("Loaded optimized hyperparameters from 'best_pso_params.pkl' (PSO output):")
    print(" OPT_ALPHA   =", OPT_ALPHA)
    print(" OPT_GAMMA   =", OPT_GAMMA)
    print(" OPT_DECAY   =", OPT_DECAY)
    print(" OPT_MIN_EPS =", OPT_MIN_EPS)
else:
    # Fallback 
    OPT_ALPHA   = 1.0
    OPT_GAMMA   = 0.8000
    OPT_DECAY   = 0.99500
    OPT_MIN_EPS = 0.010
    print("WARNING: 'best_pso_params.pkl' not found, using default optimized params.")

# Final destinations for optimized brains
FINAL_DESTINATIONS = {
    "SORTING":  (8, 5),
    "GATE_A":   (19, 3),
    "GATE_B":   (20, 16),
    "GATE_C":   (20, 31),   # updated
    "CLAIM":    (8, 32),
    "CHECKIN":  (2, 2),
}

def train_optimized(target_pos, name, episodes=8000):
    """
    Optimized Q-learning:
      - Same base rewards as baseline (+100 goal, -1 step, -10 collision)
      - PLUS distance-based shaping: moving closer to target gives a small bonus,
        moving away gives a small extra penalty.
      - More episodes than baseline.
    """
    print(f"Training Optimized Brain: {name}...")
    rows, cols = airport_grid_v11.shape
    q_table = np.zeros((rows, cols, 4))
    epsilon = 1.0

    for episode in range(episodes):
        # random non-blocked start
        while True:
            r, c = random.randint(0, rows-1), random.randint(0, cols-1)
            if airport_grid_v11[r, c] not in BLOCKED_TILES:
                break

        done = False
        steps = 0
        while not done and steps < 200:
            if random.random() < epsilon:
                action = random.randint(0, 3)
            else:
                action = int(np.argmax(q_table[r, c]))

            dr, dc = [(-1,0), (1,0), (0,-1), (0,1)][action]
            nr, nc = r + dr, c + dc

            # distance before move
            old_dist = abs(r - target_pos[0]) + abs(c - target_pos[1])

            if (nr < 0 or nr >= rows or
                nc < 0 or nc >= cols or
                airport_grid_v11[nr, nc] in BLOCKED_TILES):
                reward = -10
                nr, nc = r, c
                new_dist = old_dist
            elif (nr, nc) == target_pos:
                reward = 100
                done = True
                new_dist = 0
            else:
                new_dist = abs(nr - target_pos[0]) + abs(nc - target_pos[1])
                reward = -1
                delta_d = old_dist - new_dist
                reward += 0.5 * delta_d

            old_val = q_table[r, c, action]
            next_max = np.max(q_table[nr, nc])
            q_table[r, c, action] = (1 - OPT_ALPHA)*old_val + OPT_ALPHA*(reward + OPT_GAMMA*next_max)

            r, c = nr, nc
            steps += 1

        epsilon = max(OPT_MIN_EPS, epsilon * OPT_DECAY)

    print(f"Finished optimized training for {name}\n")
    return q_table

final_q_tables = {}
for name, pos in FINAL_DESTINATIONS.items():
    final_q_tables[name] = train_optimized(pos, name)

with open("airport_q_tables.pkl", "wb") as f:
    pickle.dump(final_q_tables, f)

print("SUCCESS: All optimized brains saved to 'airport_q_tables.pkl'")


### Cell 6 – Optimized Policy Testing (Offline)

- Loads **optimized Q-tables** from `airport_q_tables.pkl`.
- Tests **Check-in → Gate → Check-in** and **Gate → Claim** trips with greedy actions only.
- Reports **success/failure** and **number of steps**.

In [ ]:
# =========================
# CELL 6 – OPTIMIZED TESTING (offline diagnostic)
# =========================

import numpy as np
import pickle

# Final destinations
FINAL_DESTINATIONS = {
    "SORTING":  (8, 5),
    "GATE_A":   (19, 3),
    "GATE_B":   (20, 16),
    "GATE_C":   (20, 31),   
    "CLAIM":    (8, 32),
    "CHECKIN":  (2, 2),
}

with open("airport_q_tables.pkl", "rb") as f:
    optimized_tables = pickle.load(f)

def verify_trip(start_pos, name, max_steps=300):
    model = optimized_tables[name]
    target_pos = FINAL_DESTINATIONS[name]
    r, c = start_pos
    steps = 0

    while steps < max_steps:
        action = int(np.argmax(model[r, c]))
        dr, dc = [(-1,0), (1,0), (0,-1), (0,1)][action]
        nr, nc = r + dr, c + dc

        if (nr < 0 or nr >= 23 or
            nc < 0 or nc >= 35 or
            airport_grid_v11[nr, nc] in BLOCKED_TILES):
            return -1  # invalid move

        r, c = nr, nc
        steps += 1

        if (r, c) == target_pos:
            return steps

    return -1  # timeout

GATES = ["GATE_A", "GATE_B", "GATE_C"]

print("[OPTIMIZED SCENARIO 1] Departure Cycles (Check-In → Gate → Check-In)")
for gate in GATES:
    print(f"   Testing cycle for {gate}...")
    out_steps = verify_trip(FINAL_DESTINATIONS["CHECKIN"], gate)
    if out_steps != -1:
        back_steps = verify_trip(FINAL_DESTINATIONS[gate], "CHECKIN")
        if back_steps != -1:
            print(f"   PASS: Check-In -> {gate} ({out_steps} steps) -> Check-In ({back_steps} steps)")
        else:
            print(f"   FAIL: return {gate} -> Check-In failed.")
    else:
        print(f"   FAIL: outbound Check-In -> {gate} failed.")

print("\n[OPTIMIZED SCENARIO 2] Arrival Cycles (Gate → Claim)")
for gate in GATES:
    print(f"   Testing arrival from {gate}...")
    steps_arr = verify_trip(FINAL_DESTINATIONS[gate], "CLAIM")
    if steps_arr != -1:
        print(f"   PASS: {gate} -> Claim ({steps_arr} steps)")
    else:
        print(f"   FAIL: {gate} -> Claim failed.")


### Cell 7 – GWO Hyperparameter Optimization (Swarm Mode)

- Uses **Grey Wolf Optimizer (GWO)** to tune Swarm Q-learning hyperparameters  
  (α, γ, ε-decay, min-ε).
- Evaluates each “wolf” using distance-shaped rewards and average return over the last episodes.
- Tracks the **global-best Swarm parameters** and saves them into **`best_gwo_swarm_params.pkl`**.

In [ ]:
# =========================
# CELL 7 – GWO SWARM OPTIMIZATION (Grey Wolf Optimizer for Swarm hyperparameters)
# =========================

import numpy as np
import random
import pickle

# Search space 
HYPERPARAM_SPACE_SWARM = {
    "alpha":         (0.1, 1.0),
    "gamma":         (0.80, 0.999),
    "epsilon_decay": (0.995, 0.9999),
    "min_epsilon":   (0.01, 0.2),
}

# Destinations 
DESTINATIONS_SWARM = {
    "SORTING":  (8, 5),
    "GATE_A":   (19, 3),
    "GATE_B":   (20, 16),
    "GATE_C":   (20, 31),
    "CLAIM":    (8, 32),
    "CHECKIN":  (2, 2),
}

def evaluate_swarm_hyperparameters(params):
    """
    Evaluate a set of hyperparameters for Swarm Q-learning.

    params = [alpha, gamma, eps_decay, min_eps]

    We train Q-tables to reach each gate from random valid starting positions,
    using distance-shaped rewards (same style as optimized Q-learning).
    The fitness is the average return over the last episodes for all gates.
    """
    alpha, gamma, eps_decay, min_eps = params

    test_targets = ["GATE_A", "GATE_B", "GATE_C"]
    episodes    = 800
    max_steps   = 200

    rows, cols = airport_grid_v11.shape
    accumulated_score = 0.0

    for target_name in test_targets:
        target_pos = DESTINATIONS_SWARM[target_name]
        q_table = np.zeros((rows, cols, 4))
        epsilon = 1.0
        total_rewards = []

        for ep in range(episodes):
            # random non-blocked start
            while True:
                r, c = random.randint(0, rows - 1), random.randint(0, cols - 1)
                if airport_grid_v11[r, c] not in BLOCKED_TILES:
                    break

            done = False
            ep_reward = 0.0

            for step in range(max_steps):
                # epsilon-greedy
                if random.random() < epsilon:
                    action = random.randint(0, 3)
                else:
                    action = int(np.argmax(q_table[r, c]))

                dr, dc = [(-1,0), (1,0), (0,-1), (0,1)][action]
                nr, nc = r + dr, c + dc

                old_dist = abs(r - target_pos[0]) + abs(c - target_pos[1])

                if (nr < 0 or nr >= rows or
                    nc < 0 or nc >= cols or
                    airport_grid_v11[nr, nc] in BLOCKED_TILES):
                    # invalid / collision
                    reward = -10
                    nr, nc = r, c
                    new_dist = old_dist
                elif (nr, nc) == target_pos:
                    reward = 100
                    done = True
                    new_dist = 0
                else:
                    new_dist = abs(nr - target_pos[0]) + abs(nc - target_pos[1])
                    reward = -1
                    delta_d = old_dist - new_dist  # >0 if we got closer
                    reward += 0.5 * delta_d

                old_val  = q_table[r, c, action]
                next_max = np.max(q_table[nr, nc])
                q_table[r, c, action] = (1.0 - alpha) * old_val + alpha * (reward + gamma * next_max)

                r, c = nr, nc
                ep_reward += reward
                if done:
                    break

            # use eps_decay & min_eps from params
            epsilon = max(min_eps, epsilon * eps_decay)
            total_rewards.append(ep_reward)

        # Use average of last 50 episodes for this gate
        accumulated_score += np.mean(total_rewards[-50:])

    # Average across gates
    return accumulated_score / len(test_targets)

def sample_random_swarm_params():
    """Sample one random point from the Swarm hyperparameter space."""
    a = np.random.uniform(*HYPERPARAM_SPACE_SWARM["alpha"])
    g = np.random.uniform(*HYPERPARAM_SPACE_SWARM["gamma"])
    d = np.random.uniform(*HYPERPARAM_SPACE_SWARM["epsilon_decay"])
    m = np.random.uniform(*HYPERPARAM_SPACE_SWARM["min_epsilon"])
    return np.array([a, g, d, m], dtype=float)

def clamp_swarm_params(vec):
    """Clamp a position vector to the valid Swarm hyperparameter bounds."""
    vec[0] = np.clip(vec[0], *HYPERPARAM_SPACE_SWARM["alpha"])
    vec[1] = np.clip(vec[1], *HYPERPARAM_SPACE_SWARM["gamma"])
    vec[2] = np.clip(vec[2], *HYPERPARAM_SPACE_SWARM["epsilon_decay"])
    vec[3] = np.clip(vec[3], *HYPERPARAM_SPACE_SWARM["min_epsilon"])
    return vec

def GWO_swarm_optimize(num_wolves=8, iterations=8):
    """
    Grey Wolf Optimizer for Swarm Q-learning hyperparameters.
    Maximizes the fitness returned by evaluate_swarm_hyperparameters().
    """
    print("=== GWO Optimization for Swarm Hyperparameters ===\n")

    dim = 4
    wolves = np.zeros((num_wolves, dim), dtype=float)

    # initialize wolves randomly
    for i in range(num_wolves):
        wolves[i] = sample_random_swarm_params()

    # evaluate initial pack
    fitness = np.array([evaluate_swarm_hyperparameters(w) for w in wolves])

    # identify initial alpha, beta, delta 
    idx_sorted = np.argsort(-fitness)
    alpha = wolves[idx_sorted[0]].copy()
    beta  = wolves[idx_sorted[1]].copy()
    delta = wolves[idx_sorted[2]].copy()

    alpha_fit = fitness[idx_sorted[0]]

    # GLOBAL best trackers 
    global_best_pos = alpha.copy()
    global_best_fit = alpha_fit

    # main GWO loop
    for it in range(iterations):
        # coefficient a decreases linearly from 2 -> 0
        a = 2 - 2 * (it / float(iterations - 1 if iterations > 1 else 1))
        print(f"Iteration {it+1}/{iterations} (a={a:.3f})")

        for i in range(num_wolves):
            X = wolves[i].copy()

            # update relative to alpha, beta, delta
            r1, r2 = np.random.rand(dim), np.random.rand(dim)
            A1 = 2 * a * r1 - a
            C1 = 2 * r2
            D_alpha = np.abs(C1 * alpha - X)
            X1 = alpha - A1 * D_alpha

            r1, r2 = np.random.rand(dim), np.random.rand(dim)
            A2 = 2 * a * r1 - a
            C2 = 2 * r2
            D_beta = np.abs(C2 * beta - X)
            X2 = beta - A2 * D_beta

            r1, r2 = np.random.rand(dim), np.random.rand(dim)
            A3 = 2 * a * r1 - a
            C3 = 2 * r2
            D_delta = np.abs(C3 * delta - X)
            X3 = delta - A3 * D_delta

            new_pos = (X1 + X2 + X3) / 3.0
            new_pos = clamp_swarm_params(new_pos)

            wolves[i] = new_pos
            fitness[i] = evaluate_swarm_hyperparameters(new_pos)

        # re-rank wolves
        idx_sorted = np.argsort(-fitness)
        alpha = wolves[idx_sorted[0]].copy()
        beta  = wolves[idx_sorted[1]].copy()
        delta = wolves[idx_sorted[2]].copy()

        alpha_fit = fitness[idx_sorted[0]]

        # update global best if needed
        if alpha_fit > global_best_fit:
            global_best_fit = alpha_fit
            global_best_pos = alpha.copy()

        print(f"  Iteration best avg reward: {alpha_fit:.2f}")
        print(f"  Global best so far:       {global_best_fit:.2f}")
        print(f"  Global best params:       {global_best_pos}\n")

    print("=== GWO Swarm Optimization Finished ===")
    print("Best Swarm Hyperparameters (global alpha wolf):")
    print(f"alpha:         {global_best_pos[0]:.4f}")
    print(f"gamma:         {global_best_pos[1]:.4f}")
    print(f"epsilon_decay: {global_best_pos[2]:.5f}")
    print(f"min_epsilon:   {global_best_pos[3]:.3f}")
    print(f"Avg Reward:    {global_best_fit:.2f}")

    return global_best_pos, global_best_fit

best_swarm_params, best_swarm_reward = GWO_swarm_optimize()

# === SAVE BEST SWARM PARAMS ===
with open("best_gwo_swarm_params.pkl", "wb") as f:
    pickle.dump(best_swarm_params, f)

print("\nSaved best GWO Swarm parameters to 'best_gwo_swarm_params.pkl':")
print(" alpha        =", best_swarm_params[0])
print(" gamma        =", best_swarm_params[1])
print(" epsilon_decay=", best_swarm_params[2])
print(" min_epsilon  =", best_swarm_params[3])


### Cell 8 – Swarm Q-Learning Training (Using GWO Params)

- Loads the best Swarm hyperparameters from **`best_gwo_swarm_params.pkl`**.
- Trains **Swarm Q-tables** for all destinations with the same shaped reward structure.
- Saves the Swarm brains into **`airport_swarm_q_tables.pkl`**.

In [ ]:
# =========================
# CELL 8 – TRAIN SWARM BRAINS (USING BEST GWO PARAMS)
# =========================

import numpy as np
import random
import pickle
import os

if os.path.exists("best_gwo_swarm_params.pkl"):
    with open("best_gwo_swarm_params.pkl", "rb") as f:
        swarm_params = pickle.load(f)

    if len(swarm_params) != 4:
        raise ValueError("best_gwo_swarm_params.pkl must contain [alpha, gamma, epsilon_decay, min_epsilon]")

    SWARM_ALPHA, SWARM_GAMMA, SWARM_DECAY, SWARM_MIN_EPS = swarm_params
    print("Loaded Swarm hyperparameters from 'best_gwo_swarm_params.pkl':")
    print(" SWARM_ALPHA   =", SWARM_ALPHA)
    print(" SWARM_GAMMA   =", SWARM_GAMMA)
    print(" SWARM_DECAY   =", SWARM_DECAY)
    print(" SWARM_MIN_EPS =", SWARM_MIN_EPS)
else:
    SWARM_ALPHA   = 0.9
    SWARM_GAMMA   = 0.95
    SWARM_DECAY   = 0.996
    SWARM_MIN_EPS = 0.02
    print("WARNING: 'best_gwo_swarm_params.pkl' not found, using fallback Swarm params.")

SWARM_FINAL_DESTINATIONS = {
    "SORTING":  (8, 5),
    "GATE_A":   (19, 3),
    "GATE_B":   (20, 16),
    "GATE_C":   (20, 31),
    "CLAIM":    (8, 32),
    "CHECKIN":  (2, 2),
}

def train_swarm_brain(target_pos, name, episodes=8000):
    """
    Swarm Q-learning training.
    Very similar to optimized training but uses GWO-tuned Swarm hyperparameters.
    """
    print(f"Training Swarm Brain: {name}...")
    rows, cols = airport_grid_v11.shape
    q_table = np.zeros((rows, cols, 4))
    epsilon = 1.0

    for episode in range(episodes):
        while True:
            r, c = random.randint(0, rows-1), random.randint(0, cols-1)
            if airport_grid_v11[r, c] not in BLOCKED_TILES:
                break

        done = False
        steps = 0
        while not done and steps < 200:
            if random.random() < epsilon:
                action = random.randint(0, 3)
            else:
                action = int(np.argmax(q_table[r, c]))

            dr, dc = [(-1,0), (1,0), (0,-1), (0,1)][action]
            nr, nc = r + dr, c + dc

            old_dist = abs(r - target_pos[0]) + abs(c - target_pos[1])

            if (nr < 0 or nr >= rows or
                nc < 0 or nc >= cols or
                airport_grid_v11[nr, nc] in BLOCKED_TILES):
                reward = -10
                nr, nc = r, c
                new_dist = old_dist
            elif (nr, nc) == target_pos:
                reward = 100
                done = True
                new_dist = 0
            else:
                new_dist = abs(nr - target_pos[0]) + abs(nc - target_pos[1])
                reward = -1
                delta_d = old_dist - new_dist
                reward += 0.5 * delta_d

            old_val = q_table[r, c, action]
            next_max = np.max(q_table[nr, nc])
            q_table[r, c, action] = (1 - SWARM_ALPHA)*old_val + SWARM_ALPHA*(reward + SWARM_GAMMA*next_max)

            r, c = nr, nc
            steps += 1

        epsilon = max(SWARM_MIN_EPS, epsilon * SWARM_DECAY)

    print(f"Finished Swarm training for {name}\n")
    return q_table

swarm_q_tables = {}
for name, pos in SWARM_FINAL_DESTINATIONS.items():
    swarm_q_tables[name] = train_swarm_brain(pos, name)

with open("airport_swarm_q_tables.pkl", "wb") as f:
    pickle.dump(swarm_q_tables, f)

print("SUCCESS: All Swarm brains saved to 'airport_swarm_q_tables.pkl'")


### Cell 9 – Swarm Policy Testing

- Loads **Swarm Q-tables** from `airport_swarm_q_tables.pkl`.
- Tests **Check-in → Gate → Check-in** and **Gate → Claim** cycles using greedy Swarm policies.
- Prints success status and **step counts**.


In [ ]:
# =========================
# CELL 9 – SWARM TESTING
# =========================

import numpy as np
import pickle

SWARM_FINAL_DESTINATIONS = {
    "SORTING":  (8, 5),
    "GATE_A":   (19, 3),
    "GATE_B":   (20, 16),
    "GATE_C":   (20, 31),
    "CLAIM":    (8, 32),
    "CHECKIN":  (2, 2),
}

with open("airport_swarm_q_tables.pkl", "rb") as f:
    swarm_tables = pickle.load(f)

def verify_swarm_trip(start_pos, name, max_steps=300):
    model = swarm_tables[name]
    target_pos = SWARM_FINAL_DESTINATIONS[name]
    r, c = start_pos
    steps = 0

    while steps < max_steps:
        action = int(np.argmax(model[r, c]))
        dr, dc = [(-1,0), (1,0), (0,-1), (0,1)][action]
        nr, nc = r + dr, c + dc

        if (nr < 0 or nr >= 23 or
            nc < 0 or nc >= 35 or
            airport_grid_v11[nr, nc] in BLOCKED_TILES):
            return -1

        r, c = nr, nc
        steps += 1

        if (r, c) == target_pos:
            return steps

    return -1

GATES = ["GATE_A", "GATE_B", "GATE_C"]

print("[SWARM (GWO) SCENARIO 1] Departure Cycles (Check-In → Gate → Check-In)")
for gate in GATES:
    print(f"   Testing cycle for {gate}...")
    out_steps = verify_swarm_trip(SWARM_FINAL_DESTINATIONS["CHECKIN"], gate)
    if out_steps != -1:
        back_steps = verify_swarm_trip(SWARM_FINAL_DESTINATIONS[gate], "CHECKIN")
        if back_steps != -1:
            print(f"   PASS: Check-In -> {gate} ({out_steps} steps) -> Check-In ({back_steps} steps)")
        else:
            print(f"   FAIL: return {gate} -> Check-In failed.")
    else:
        print(f"   FAIL: outbound Check-In -> {gate} failed.")

print("\n[SWARM (GWO) SCENARIO 2] Arrival Cycles (Gate → Claim)")
for gate in GATES:
    print(f"   Testing arrival from {gate}...")
    steps_arr = verify_swarm_trip(SWARM_FINAL_DESTINATIONS[gate], "CLAIM")
    if steps_arr != -1:
        print(f"   PASS: {gate} -> Claim ({steps_arr} steps)")
    else:
        print(f"   FAIL: {gate} -> Claim failed.")


### Cell 10 – Key Scenarios for Each Mode Test

- Loads **Baseline, Optimized, and Swarm** Q-tables and runs **greedy traces** for key scenarios  
  (Check-in → Gate, Gate → Claim).
- For each mode and scenario, measures:
  - success/failure reason, number of **steps**, **revisits**, and **path length**.
- Used as a **diagnostic tool** to detect loops, bad routes, and differences between the three policies.

In [ ]:
# =========================
# CELL 10 – Each Mode Key Scenarios
# =========================

import numpy as np
import pickle

# Destinations (must match the environment)
DESTINATIONS = {
    "SORTING":  (8, 5),
    "GATE_A":   (19, 3),
    "GATE_B":   (20, 16),
    "GATE_C":   (20, 31),
    "CLAIM":    (8, 32),
    "CHECKIN":  (2, 2),
}

# Load Q-tables for all three modes
with open("airport_q_baseline.pkl", "rb") as f:
    diag_baseline_tables = pickle.load(f)

with open("airport_q_tables.pkl", "rb") as f:
    diag_optimized_tables = pickle.load(f)

try:
    with open("airport_swarm_q_tables.pkl", "rb") as f:
        diag_swarm_tables = pickle.load(f)
except FileNotFoundError:
    diag_swarm_tables = None
    print("WARNING: 'airport_q_swarm.pkl' not found. Swarm diagnostics will be skipped.\n")

def trace_policy(q_tables, start_pos, goal_key, max_steps=300):
    """
    Follow greedy policy (argmax) from start_pos to FINAL_DESTINATIONS[goal_key].
    Returns dict with:
      success (bool), steps, revisits, reason, path_length
    """
    model = q_tables[goal_key]
    target_pos = DESTINATIONS[goal_key]
    r, c = start_pos
    steps = 0
    visited_counts = {}
    path = [(r, c)]

    while steps < max_steps:
        steps += 1
        action = int(np.argmax(model[r, c]))
        dr, dc = [(-1,0), (1,0), (0,-1), (0,1)][action]
        nr, nc = r + dr, c + dc

        if (nr < 0 or nr >= GRID_HEIGHT or
            nc < 0 or nc >= GRID_WIDTH or
            airport_grid_v11[nr, nc] in BLOCKED_TILES):
            return {
                "success": False,
                "steps": steps,
                "revisits": sum(max(0, v-1) for v in visited_counts.values()),
                "reason": "invalid_move",
                "path_len": len(path),
            }

        r, c = nr, nc
        path.append((r, c))
        visited_counts[(r, c)] = visited_counts.get((r, c), 0) + 1

        if (r, c) == target_pos:
            return {
                "success": True,
                "steps": steps,
                "revisits": sum(max(0, v-1) for v in visited_counts.values()),
                "reason": "goal",
                "path_len": len(path),
            }

    return {
        "success": False,
        "steps": steps,
        "revisits": sum(max(0, v-1) for v in visited_counts.values()),
        "reason": "timeout",
        "path_len": len(path),
    }

SCENARIOS = [
    ("CHECKIN", "GATE_A"),
    ("CHECKIN", "GATE_B"),
    ("CHECKIN", "GATE_C"),
    ("GATE_A",  "CLAIM"),
    ("GATE_B",  "CLAIM"),
    ("GATE_C",  "CLAIM"),
]

def run_mode_diagnostics(mode_name, tables):
    print(f"\n=== {mode_name} OFFLINE TRACES ===")
    for start_key, goal_key in SCENARIOS:
        res = trace_policy(tables, DESTINATIONS[start_key], goal_key)
        status = "OK" if res["success"] else f"FAIL ({res['reason']})"
        print(
            f"{start_key:8s} -> {goal_key:6s} | "
            f"status={status:12s} | steps={res['steps']:3d} | "
            f"revisits={res['revisits']:3d} | path_len={res['path_len']:3d}"
        )

run_mode_diagnostics("BASELINE", diag_baseline_tables)
run_mode_diagnostics("OPTIMIZED", diag_optimized_tables)
if diag_swarm_tables is not None:
    run_mode_diagnostics("SWARM", diag_swarm_tables)
